In [1]:
"""
╔════════════════════════════════════════════════════════════════════════════╗
║  🎯 CELLULE DE CONFIGURATION STANDALONE - COPIER-COLLER DANS VOS NOTEBOOKS ║
╚════════════════════════════════════════════════════════════════════════════╝

INSTRUCTIONS:
-------------
1. Copiez TOUT le contenu de cette cellule
2. Collez-le comme PREMIÈRE CELLULE de votre notebook
3. Exécutez la cellule
4. La configuration est prête à l'emploi !

Cette cellule est 100% autonome et fonctionne partout :
✅ Google Colab (clone + installe automatiquement)
✅ WSL / Linux Local
✅ Tout environnement Jupyter

APRÈS EXÉCUTION, UTILISEZ L'OBJET 'config':
--------------------------------------------
▶ config.data_dir              # Chemin du dataset
▶ config.models_dir            # Répertoire des modèles
▶ config.results_dir           # Répertoire des résultats
▶ config.classes               # Liste des classes
▶ config.img_size              # Tuple (width, height)
▶ config.img_channels          # Nombre de canaux (1=grayscale, 3=RGB)
▶ config.batch_size            # Taille des batchs
▶ config.epochs                # Nombre d'époques
▶ config.learning_rate         # Learning rate
▶ config.validation_split      # Proportion pour validation
▶ config.gradcam_alpha         # Alpha pour Grad-CAM
▶ config.shap_max_evals        # Evaluations SHAP
▶ config.confidence_high_threshold  # Seuil confiance haute
... et bien plus !

VARIABLES GLOBALES:
-------------------
• config: Objet Config complet (tous les paramètres du projet)
• ENV: Environnement détecté ('colab', 'wsl', 'local')
• Tous les transformers importés et prêts à l'emploi

"""

# =============================================================================
# IMPORTS STANDARDS
# =============================================================================

import os
import sys
import subprocess
from pathlib import Path


# =============================================================================
# DÉTECTION AUTOMATIQUE DE L'ENVIRONNEMENT
# =============================================================================

def detect_environment():
    """Détecte l'environnement (colab, wsl, local)"""
    try:
        import google.colab
        return "colab"
    except ImportError:
        is_wsl = os.path.exists('/proc/version') and 'microsoft' in open('/proc/version').read().lower()
        return "wsl" if is_wsl else "local"

ENV = detect_environment()
print(f"🌍 Environnement: {ENV.upper()}")


# =============================================================================
# BOOTSTRAP COLAB (Clone + Install si nécessaire)
# =============================================================================

if ENV == "colab":
    print("\n🚀 Bootstrap Colab...")
    
    os.chdir('/content')
    if not os.path.exists('/content/Data_Pipeline'):
        print("📥 Clonage du repository...")
        subprocess.run(['git', 'clone', 'https://github.com/L-Poca/Data_Pipeline.git'], check=True)
    
    os.chdir('/content/Data_Pipeline')
    
    # Checkout de la branche rafael_cleaning
    result = subprocess.run(
        ['git', 'checkout', '-b', 'rafael_cleaning', 'origin/rafael_cleaning'],
        capture_output=True,
        text=True
    )
    if result.returncode != 0:
        # Si la branche locale existe déjà, juste switcher
        subprocess.run(['git', 'checkout', 'rafael_cleaning'], capture_output=True)
    
    # Installation du package en mode éditable (sans dépendances - détection Colab dans setup.py)
    print("📦 Installation du package...")
    result = subprocess.run(['pip', 'install', '-e', '.', '--quiet'], capture_output=True, text=True)
    if result.returncode != 0:
        print(f"⚠️ Erreur installation: {result.stderr}")
    else:
        print("✅ Package installé")
    
    print("💾 Montage Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Extraction dataset
    archive_data = '/content/drive/MyDrive/DS_COVID/archive_covid.zip'
    if os.path.exists(archive_data):
        print("📦 Extraction dataset...")
        os.makedirs('./data/raw/', exist_ok=True)
        subprocess.run(['unzip', '-o', '-q', archive_data, '-d', './data/raw/COVID-19_Radiography_Dataset/'])
    
    # Extraction models
    archive_models = '/content/drive/MyDrive/DS_COVID/inceptionv3_best.zip'
    if os.path.exists(archive_models):
        print("📦 Extraction models...")
        os.makedirs('./models/', exist_ok=True)
        subprocess.run(['unzip', '-o', '-q', archive_models, '-d', './models/'])

    print("✅ Bootstrap terminé")


# =============================================================================
# CONFIGURATION DES CHEMINS
# =============================================================================

# Déterminer project_root selon l'environnement
if ENV == "colab":
    project_root = Path('/content/Data_Pipeline')
elif ENV == "wsl":
    project_root = Path('/home/cepa/DST/projet_DS/Data_Pipeline/Data_Pipeline')
else:  # local
    # Depuis un notebook dans src/notebooks/
    project_root = Path.cwd().parent.parent

# Vérification du modèle en local (WSL ou autre)
if ENV != "colab":
    models_dir = project_root / 'models'
    model_path = models_dir / 'inceptionv3_best.keras'
    
    if model_path.exists():
        print(f"✅ Modèle InceptionV3 trouvé: {model_path}")
    else:
        print(f"⚠️ Modèle InceptionV3 non trouvé: {model_path}")
        print(f"   Veuillez placer inceptionv3_best.keras dans {models_dir}/")

# Ajouter src/ au sys.path pour les imports
# src_path = str(project_root / 'src')
# if src_path not in sys.path:
#     sys.path.insert(0, src_path)
#     print(f"✅ Chemin src/ ajouté: {src_path}")

# Charger la configuration depuis JSON
from src.utils.config import build_config

config = build_config(project_root, ENV)

print(f"\n🎯 Configuration chargée depuis config/{ENV}_config.json")


# =============================================================================
# IMPORTS DES TRANSFORMERS
# =============================================================================

try:
    from src.features.Pipelines.Transformateurs.image_loaders import ImageLoader
    from src.features.Pipelines.Transformateurs.image_preprocessing import (
        ImageResizer, ImageNormalizer, ImageFlattener, ImageMasker
    )
    from src.features.Pipelines.Transformateurs.image_augmentation import (
        ImageAugmenter, ImageRandomCropper
    )
    from src.features.Pipelines.Transformateurs.image_features import (
        ImageHistogram, ImagePCA, ImageStandardScaler
    )
    print("✅ Transformers importés")
except ImportError as e:
    print(f"⚠️ Erreur import transformers: {e}")


# =============================================================================
# IMPORTS ML/DL
# =============================================================================

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras

# =============================================================================
# CONFIGURATION MATPLOTLIB (utilise config pour les paramètres)
# =============================================================================

plt.rcParams['figure.figsize'] = config.figure_size
plt.rcParams['figure.dpi'] = config.dpi
plt.style.use(config.plot_style)
sns.set_palette(config.color_palette)

# =============================================================================
# AFFICHAGE DU RÉSUMÉ
# =============================================================================

print("\n" + "=" * 80)
print("✅ CONFIGURATION PRÊTE - Data Pipeline")
print("=" * 80)
print(f"📂 Projet:       {config.project_root}")
print(f"📊 Dataset:      {config.data_dir}")
print(f"💾 Modèles:      {config.models_dir}")
print(f"📈 Résultats:    {config.results_dir}")
print(f"📐 Dataset:      {'✅ Accessible' if config.data_dir.exists() else '❌ Introuvable'}")
print()
print(f"🏷️  Classes:     {', '.join(config.classes)} ({config.num_classes} classes)")
print(f"🎛️  Images:      {config.img_size} | {config.img_channels} canaux")
print(f"🔧 Training:     Batch={config.batch_size} | Epochs={config.epochs} | LR={config.learning_rate}")
print(f"� Splits:       Train/Val={1-config.validation_split:.0%} | Val={config.validation_split:.0%} | Test={config.test_split:.0%}")
print()
print(f"🎨 Viz:          Style={config.plot_style} | Palette={config.color_palette}")
print(f"📏 Figures:      {config.figure_size} @ {config.dpi} DPI")
print()
print(f"🔍 Interprét.:   GradCAM α={config.gradcam_alpha} | SHAP evals={config.shap_max_evals}")
print(f"📉 Seuils conf.: High={config.confidence_high_threshold} | Medium={config.confidence_medium_threshold}")
print("=" * 80)
print("\n💡 Variable principale:")
print("   • config: Objet Config complet (accès à TOUS les paramètres)")
print("   • ENV: Environnement actuel")
print()
print("📚 Exemples d'utilisation:")
print("   config.data_dir          # Chemin du dataset")
print("   config.classes           # Liste des classes")
print("   config.img_size          # Tuple (width, height)")
print("   config.batch_size        # Taille des batchs")
print("   config.models_dir        # Répertoire des modèles")
print("   config.gradcam_alpha     # Paramètres d'interprétabilité")
print()
print("🎯 Transformers disponibles:")
print("   • ImageLoader, ImageResizer, ImageNormalizer, ImageFlattener, ImageMasker")
print("   • ImageAugmenter, ImageRandomCropper")
print("   • ImageHistogram, ImagePCA, ImageStandardScaler")
print("=" * 80)


🌍 Environnement: WSL
✅ Modèle InceptionV3 trouvé: /home/cepa/DST/projet_DS/Data_Pipeline/Data_Pipeline/models/inceptionv3_best.keras

🎯 Configuration chargée depuis config/wsl_config.json
✅ Transformers importés


2025-11-08 13:29:09.921590: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-11-08 13:29:09.926262: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-08 13:29:10.078031: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-08 13:29:12.198421: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off,


✅ CONFIGURATION PRÊTE - Data Pipeline
📂 Projet:       /home/cepa/DST/projet_DS/Data_Pipeline/Data_Pipeline
📊 Dataset:      /home/cepa/DST/projet_DS/Data_Pipeline/Data_Pipeline/data/raw/COVID-19_Radiography_Dataset/COVID-19_Radiography_Dataset
💾 Modèles:      /home/cepa/DST/projet_DS/Data_Pipeline/Data_Pipeline/models
📈 Résultats:    /home/cepa/DST/projet_DS/Data_Pipeline/Data_Pipeline/results
📐 Dataset:      ✅ Accessible

🏷️  Classes:     COVID, Normal, Lung_Opacity, Viral Pneumonia (4 classes)
🎛️  Images:      (256, 256) | 1 canaux
🔧 Training:     Batch=128 | Epochs=50 | LR=0.001
� Splits:       Train/Val=85% | Val=15% | Test=15%

🎨 Viz:          Style=seaborn-v0_8 | Palette=husl
📏 Figures:      (12, 8) @ 100 DPI

🔍 Interprét.:   GradCAM α=0.4 | SHAP evals=100
📉 Seuils conf.: High=0.8 | Medium=0.6

💡 Variable principale:
   • config: Objet Config complet (accès à TOUS les paramètres)
   • ENV: Environnement actuel

📚 Exemples d'utilisation:
   config.data_dir          # Chemin du dat

# 🦠 Advanced Exploratory Data Analysis (EDA) - COVID-19 Radiography Dataset

## 📊 Overview
Ce notebook présente une analyse exploratoire avancée du dataset COVID-19 Radiography.

### 🎯 Objectifs:
- Analyse statistique approfondie des images et masques
- Réduction de dimensionnalité (PCA, t-SNE)
- Clustering automatique (K-Means, DBSCAN, Hierarchical)
- Tests statistiques et corrélations
- Génération de rapport HTML automatique

### 📚 Dataset:
- **Classes**: COVID, Normal, Lung_Opacity, Viral Pneumonia
- **Images**: Radiographies thoraciques
- **Masques**: Segmentation des régions d'intérêt

---

In [2]:
# =============================================================================
# IMPORTS POUR EDA AVANCÉE
# =============================================================================

# Dimensionality Reduction & Manifold Learning
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# Clustering
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from scipy.cluster.hierarchy import dendrogram, linkage

# Statistical Analysis
from scipy import stats
from scipy.stats import chi2_contingency, f_oneway, kruskal

# Feature Extraction
from skimage.feature import graycomatrix, graycoprops, local_binary_pattern
from skimage import exposure

# Visualization
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Utils
from tqdm.notebook import tqdm
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("✅ Imports pour EDA avancée chargés")

✅ Imports pour EDA avancée chargés


## 1. 📥 Chargement des Données

Chargement des images radiographiques et des masques de segmentation pour chaque classe.

In [4]:
# =============================================================================
# CHARGEMENT DU DATASET (Images + Masques)
# =============================================================================

from src.notebooks import load_dataset, create_preprocessing_pipeline

# Charger les chemins des images et masques
# Note: load_masks=True pour charger aussi les masques de segmentation
image_paths, mask_paths, labels, labels_int = load_dataset(
    data_dir=config.data_dir,
    categories=config.classes,
    n_images_per_class=None,  # Limite pour économiser la mémoire
    load_masks=True,  # ⚠️ Important: charge aussi les masques
    verbose=True
)

# Créer une pipeline de preprocessing pour les images
# Cette pipeline inclut: ImageLoader, ImageResizer, ImageMasker
pipeline_img = create_preprocessing_pipeline(
    img_size=config.img_size,
    color_mode='L',  # Grayscale pour les radiographies
    mask_paths=mask_paths if len(mask_paths) > 0 else None,
    verbose=True
)

# Charger et preprocesser les images
print("\n📊 Preprocessing des images...")
images = pipeline_img.fit_transform(image_paths)

# Normaliser les images [0, 1]
images = images.astype('float32') / 255.0

print(f"\n✅ Dataset chargé:")
print(f"   - Images shape: {images.shape}")
print(f"   - Labels shape: {labels_int.shape}")
print(f"   - Nombre de masques: {len(mask_paths)}")
print(f"   - Classes: {config.classes}")

CHARGEMENT DES DONNÉES
  COVID               : 3616 images (avec masques)
  Normal              : 10192 images (avec masques)
  Lung_Opacity        : 6012 images (avec masques)
  Viral Pneumonia     : 1345 images (avec masques)

  Total: 21165 images
  Classes: 4
  Distribution: [ 3616 10192  6012  1345]

✅ Chemins des masques récupérés: 21165
PREPROCESSING PIPELINE

✅ Pipeline créée avec 3 étapes

📊 Preprocessing des images...


Resizing images: 100%|██████████| 21165/21165 [00:15<00:00, 1367.00it/s]


: 

## 2. 📊 Analyse Statistique de Base

Analyse de la distribution des classes et statistiques descriptives des images.

In [ ]:
# =============================================================================
# ANALYSE STATISTIQUE DE BASE
# =============================================================================

# Distribution des classes
unique_classes, class_counts = np.unique(labels_int, return_counts=True)

print("=" * 70)
print("DISTRIBUTION DES CLASSES")
print("=" * 70)
for cls_idx, count in zip(unique_classes, class_counts):
    cls_name = config.classes[cls_idx]
    percentage = (count / len(labels_int)) * 100
    print(f"{cls_name:20s}: {count:5d} images ({percentage:5.2f}%)")

# Statistiques descriptives des images
print("\n" + "=" * 70)
print("STATISTIQUES DES IMAGES (par classe)")
print("=" * 70)

stats_per_class = {}
for cls_idx, cls_name in enumerate(config.classes):
    # Sélectionner les images de cette classe
    cls_mask = labels_int == cls_idx
    cls_images = images[cls_mask]
    
    # Calculer les statistiques
    stats_per_class[cls_name] = {
        'mean': np.mean(cls_images),
        'std': np.std(cls_images),
        'min': np.min(cls_images),
        'max': np.max(cls_images),
        'median': np.median(cls_images),
        'q25': np.percentile(cls_images, 25),
        'q75': np.percentile(cls_images, 75)
    }
    
    print(f"\n{cls_name}:")
    print(f"  Mean:   {stats_per_class[cls_name]['mean']:.4f}")
    print(f"  Std:    {stats_per_class[cls_name]['std']:.4f}")
    print(f"  Min:    {stats_per_class[cls_name]['min']:.4f}")
    print(f"  Max:    {stats_per_class[cls_name]['max']:.4f}")
    print(f"  Median: {stats_per_class[cls_name]['median']:.4f}")
    print(f"  Q25:    {stats_per_class[cls_name]['q25']:.4f}")
    print(f"  Q75:    {stats_per_class[cls_name]['q75']:.4f}")

# Visualisation de la distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Distribution des classes
axes[0].bar(range(len(config.classes)), class_counts, color=sns.color_palette('husl', len(config.classes)))
axes[0].set_xticks(range(len(config.classes)))
axes[0].set_xticklabels(config.classes, rotation=45, ha='right')
axes[0].set_ylabel('Nombre d\'images')
axes[0].set_title('Distribution des Classes')
axes[0].grid(alpha=0.3)

# Box plot des intensités moyennes par classe
mean_intensities_per_class = []
for cls_idx in range(len(config.classes)):
    cls_mask = labels_int == cls_idx
    cls_images = images[cls_mask]
    # Calculer l'intensité moyenne de chaque image
    mean_intensities = np.mean(cls_images, axis=(1, 2, 3))
    mean_intensities_per_class.append(mean_intensities)

axes[1].boxplot(mean_intensities_per_class, labels=config.classes)
axes[1].set_xticklabels(config.classes, rotation=45, ha='right')
axes[1].set_ylabel('Intensité Moyenne')
axes[1].set_title('Distribution des Intensités Moyennes par Classe')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✅ Analyse statistique de base terminée")

## 3. 🔬 Analyse en Composantes Principales (PCA)

Réduction de dimensionnalité avec PCA pour identifier les composantes principales et visualiser la variance expliquée.

In [ ]:
# =============================================================================
# ANALYSE PCA (Principal Component Analysis)
# =============================================================================

print("=" * 70)
print("ANALYSE PCA")
print("=" * 70)

# Aplatir les images pour PCA (n_samples, n_features)
# De (n, h, w, c) vers (n, h*w*c)
n_samples = images.shape[0]
images_flat = images.reshape(n_samples, -1)

print(f"\n📊 Shape après aplatissement: {images_flat.shape}")
print(f"   - {n_samples} échantillons")
print(f"   - {images_flat.shape[1]} features par échantillon")

# PCA avec 50 composantes (configurable)
n_components = config.transformers['pca']['n_components']
print(f"\n🔬 Application de PCA avec {n_components} composantes...")

pca = PCA(n_components=n_components, random_state=config.training['random_seed'])
images_pca = pca.fit_transform(images_flat)

print(f"\n✅ PCA terminée:")
print(f"   - Shape réduite: {images_pca.shape}")
print(f"   - Variance expliquée totale: {pca.explained_variance_ratio_.sum():.4f}")
print(f"   - Variance par les 10 premières composantes: {pca.explained_variance_ratio_[:10].sum():.4f}")

# Afficher la variance expliquée par composante
print(f"\n📈 Variance expliquée par les 10 premières composantes:")
for i in range(min(10, n_components)):
    cumsum = pca.explained_variance_ratio_[:i+1].sum()
    print(f"   PC{i+1:2d}: {pca.explained_variance_ratio_[i]:.4f} (cumulée: {cumsum:.4f})")

# Visualisation de la variance expliquée
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Variance expliquée par composante
axes[0].bar(range(1, n_components+1), pca.explained_variance_ratio_, 
           color=sns.color_palette('viridis', 1)[0], alpha=0.7)
axes[0].set_xlabel('Composante Principale')
axes[0].set_ylabel('Variance Expliquée')
axes[0].set_title('Variance Expliquée par Composante')
axes[0].grid(alpha=0.3)
axes[0].set_xlim(0, n_components+1)

# Variance expliquée cumulée
cumsum_variance = np.cumsum(pca.explained_variance_ratio_)
axes[1].plot(range(1, n_components+1), cumsum_variance, 
            'o-', color=sns.color_palette('viridis', 1)[0], linewidth=2, markersize=4)
axes[1].axhline(y=0.95, color='r', linestyle='--', label='95% variance')
axes[1].axhline(y=0.99, color='orange', linestyle='--', label='99% variance')
axes[1].set_xlabel('Nombre de Composantes')
axes[1].set_ylabel('Variance Expliquée Cumulée')
axes[1].set_title('Variance Expliquée Cumulée')
axes[1].grid(alpha=0.3)
axes[1].legend()
axes[1].set_xlim(0, n_components+1)
axes[1].set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

# Visualiser les données PCA en 2D (PC1 vs PC2)
fig = plt.figure(figsize=(12, 8))

# Créer une colormap pour chaque classe
colors = sns.color_palette('husl', len(config.classes))
for cls_idx, cls_name in enumerate(config.classes):
    cls_mask = labels_int == cls_idx
    plt.scatter(
        images_pca[cls_mask, 0], 
        images_pca[cls_mask, 1],
        c=[colors[cls_idx]], 
        label=cls_name, 
        alpha=0.6, 
        s=30
    )

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)')
plt.title('Projection PCA 2D - COVID-19 Radiography Dataset')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Projection 3D (PC1, PC2, PC3)
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')

for cls_idx, cls_name in enumerate(config.classes):
    cls_mask = labels_int == cls_idx
    ax.scatter(
        images_pca[cls_mask, 0], 
        images_pca[cls_mask, 1],
        images_pca[cls_mask, 2],
        c=[colors[cls_idx]], 
        label=cls_name, 
        alpha=0.6, 
        s=20
    )

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%})')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%})')
ax.set_zlabel(f'PC3 ({pca.explained_variance_ratio_[2]:.2%})')
ax.set_title('Projection PCA 3D - COVID-19 Radiography Dataset')
ax.legend()
plt.tight_layout()
plt.show()

print("\n✅ Analyse PCA terminée")

## 4. 🌐 t-SNE (t-distributed Stochastic Neighbor Embedding)

Visualisation non-linéaire des données haute dimension avec t-SNE pour révéler la structure locale des données.

In [ ]:
# =============================================================================
# ANALYSE t-SNE (t-distributed Stochastic Neighbor Embedding)
# =============================================================================

print("=" * 70)
print("ANALYSE t-SNE")
print("=" * 70)

# Utiliser les données PCA pour accélérer t-SNE
# t-SNE est coûteux en calcul, donc on réduit d'abord avec PCA
print(f"\n🔬 Application de t-SNE sur les {n_components} composantes PCA...")
print(f"   Note: t-SNE peut prendre plusieurs minutes selon la taille du dataset")

# t-SNE avec différents perplexity values
perplexities = [30, 50]  # Valeurs classiques

tsne_results = {}
for perplexity in perplexities:
    print(f"\n📊 t-SNE avec perplexity={perplexity}...")
    
    tsne = TSNE(
        n_components=2,  # Projection en 2D
        perplexity=perplexity,
        n_iter=1000,
        random_state=config.training['random_seed'],
        verbose=0
    )
    
    # Appliquer t-SNE sur les données PCA
    images_tsne = tsne.fit_transform(images_pca)
    tsne_results[perplexity] = images_tsne
    
    print(f"   ✅ Shape résultante: {images_tsne.shape}")

# Visualiser les résultats t-SNE
fig, axes = plt.subplots(1, len(perplexities), figsize=(15, 6))
if len(perplexities) == 1:
    axes = [axes]

colors = sns.color_palette('husl', len(config.classes))

for idx, perplexity in enumerate(perplexities):
    images_tsne = tsne_results[perplexity]
    
    # Plot chaque classe
    for cls_idx, cls_name in enumerate(config.classes):
        cls_mask = labels_int == cls_idx
        axes[idx].scatter(
            images_tsne[cls_mask, 0],
            images_tsne[cls_mask, 1],
            c=[colors[cls_idx]],
            label=cls_name,
            alpha=0.6,
            s=30
        )
    
    axes[idx].set_xlabel('t-SNE Component 1')
    axes[idx].set_ylabel('t-SNE Component 2')
    axes[idx].set_title(f't-SNE (perplexity={perplexity})')
    axes[idx].legend(loc='best', fontsize=8)
    axes[idx].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Visualisation interactive avec Plotly (si disponible)
try:
    print("\n🎨 Création d'une visualisation interactive t-SNE avec Plotly...")
    
    # Utiliser le t-SNE avec perplexity=30
    images_tsne = tsne_results[30]
    
    # Créer un DataFrame pour Plotly
    import pandas as pd
    df_tsne = pd.DataFrame({
        't-SNE 1': images_tsne[:, 0],
        't-SNE 2': images_tsne[:, 1],
        'Classe': [config.classes[idx] for idx in labels_int],
        'Label': labels_int
    })
    
    # Créer le scatter plot interactif
    fig = px.scatter(
        df_tsne,
        x='t-SNE 1',
        y='t-SNE 2',
        color='Classe',
        title='Visualisation Interactive t-SNE - COVID-19 Dataset',
        hover_data=['Label'],
        opacity=0.7
    )
    
    fig.update_traces(marker=dict(size=5))
    fig.update_layout(
        width=900,
        height=600,
        template='plotly_white'
    )
    
    fig.show()
    print("   ✅ Visualisation interactive créée")
    
except Exception as e:
    print(f"   ⚠️ Plotly non disponible ou erreur: {e}")

print("\n✅ Analyse t-SNE terminée")

## 5. 🎯 Analyse de Clustering

Application de différents algorithmes de clustering pour identifier des groupes naturels dans les données:
- K-Means
- DBSCAN
- Hierarchical Clustering

In [ ]:
# =============================================================================
# ANALYSE DE CLUSTERING
# =============================================================================

print("=" * 70)
print("ANALYSE DE CLUSTERING")
print("=" * 70)

# Utiliser les données PCA pour le clustering (plus rapide)
X_cluster = images_pca

# 1. K-MEANS CLUSTERING
# ---------------------
print("\n1️⃣ K-Means Clustering")
print("   " + "-" * 50)

# Tester différents nombres de clusters
n_clusters_range = range(2, 8)
silhouette_scores = []
davies_bouldin_scores = []
calinski_harabasz_scores = []

print(f"   📊 Test de K-Means avec k de {n_clusters_range.start} à {n_clusters_range.stop-1}...")
for k in n_clusters_range:
    kmeans = KMeans(
        n_clusters=k,
        random_state=config.training['random_seed'],
        n_init=10
    )
    cluster_labels = kmeans.fit_predict(X_cluster)
    
    # Calculer les métriques de qualité
    sil_score = silhouette_score(X_cluster, cluster_labels)
    db_score = davies_bouldin_score(X_cluster, cluster_labels)
    ch_score = calinski_harabasz_score(X_cluster, cluster_labels)
    
    silhouette_scores.append(sil_score)
    davies_bouldin_scores.append(db_score)
    calinski_harabasz_scores.append(ch_score)

# Visualiser les métriques de clustering
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Silhouette Score (plus élevé = meilleur)
axes[0].plot(list(n_clusters_range), silhouette_scores, 'o-', linewidth=2, markersize=8)
axes[0].set_xlabel('Nombre de Clusters')
axes[0].set_ylabel('Silhouette Score')
axes[0].set_title('Silhouette Score (↑ meilleur)')
axes[0].grid(alpha=0.3)
axes[0].axvline(x=len(config.classes), color='r', linestyle='--', label=f'Nombre réel de classes ({len(config.classes)})')
axes[0].legend()

# Davies-Bouldin Index (plus bas = meilleur)
axes[1].plot(list(n_clusters_range), davies_bouldin_scores, 'o-', linewidth=2, markersize=8, color='orange')
axes[1].set_xlabel('Nombre de Clusters')
axes[1].set_ylabel('Davies-Bouldin Index')
axes[1].set_title('Davies-Bouldin Index (↓ meilleur)')
axes[1].grid(alpha=0.3)
axes[1].axvline(x=len(config.classes), color='r', linestyle='--', label=f'Nombre réel de classes ({len(config.classes)})')
axes[1].legend()

# Calinski-Harabasz Score (plus élevé = meilleur)
axes[2].plot(list(n_clusters_range), calinski_harabasz_scores, 'o-', linewidth=2, markersize=8, color='green')
axes[2].set_xlabel('Nombre de Clusters')
axes[2].set_ylabel('Calinski-Harabasz Score')
axes[2].set_title('Calinski-Harabasz Score (↑ meilleur)')
axes[2].grid(alpha=0.3)
axes[2].axvline(x=len(config.classes), color='r', linestyle='--', label=f'Nombre réel de classes ({len(config.classes)})')
axes[2].legend()

plt.tight_layout()
plt.show()

# K-Means optimal (utiliser le nombre réel de classes)
k_optimal = len(config.classes)
print(f"\n   ✅ K-Means optimal avec k={k_optimal}")

kmeans_optimal = KMeans(
    n_clusters=k_optimal,
    random_state=config.training['random_seed'],
    n_init=10
)
kmeans_labels = kmeans_optimal.fit_predict(X_cluster)

print(f"      - Silhouette Score: {silhouette_scores[k_optimal-2]:.4f}")
print(f"      - Davies-Bouldin Index: {davies_bouldin_scores[k_optimal-2]:.4f}")
print(f"      - Calinski-Harabasz Score: {calinski_harabasz_scores[k_optimal-2]:.2f}")

# Visualiser K-Means sur projection t-SNE
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Vraies classes
colors_true = sns.color_palette('husl', len(config.classes))
for cls_idx, cls_name in enumerate(config.classes):
    cls_mask = labels_int == cls_idx
    axes[0].scatter(
        tsne_results[30][cls_mask, 0],
        tsne_results[30][cls_mask, 1],
        c=[colors_true[cls_idx]],
        label=cls_name,
        alpha=0.6,
        s=30
    )
axes[0].set_xlabel('t-SNE 1')
axes[0].set_ylabel('t-SNE 2')
axes[0].set_title('Vraies Classes')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Clusters K-Means
colors_cluster = sns.color_palette('viridis', k_optimal)
for cluster_id in range(k_optimal):
    cluster_mask = kmeans_labels == cluster_id
    axes[1].scatter(
        tsne_results[30][cluster_mask, 0],
        tsne_results[30][cluster_mask, 1],
        c=[colors_cluster[cluster_id]],
        label=f'Cluster {cluster_id}',
        alpha=0.6,
        s=30
    )
axes[1].set_xlabel('t-SNE 1')
axes[1].set_ylabel('t-SNE 2')
axes[1].set_title(f'K-Means Clustering (k={k_optimal})')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✅ K-Means clustering terminé")

In [ ]:
# 2. DBSCAN CLUSTERING
# --------------------
print("\n2️⃣ DBSCAN Clustering (Density-Based)")
print("   " + "-" * 50)

# DBSCAN ne nécessite pas de spécifier le nombre de clusters
# mais nécessite eps (distance) et min_samples
dbscan = DBSCAN(
    eps=3.0,  # Distance epsilon (ajustable)
    min_samples=5,  # Nombre minimum de samples dans un voisinage
    metric='euclidean'
)

print("   📊 Application de DBSCAN...")
dbscan_labels = dbscan.fit_predict(X_cluster)

# Nombre de clusters trouvés (sans compter le bruit: -1)
n_clusters_dbscan = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_noise = list(dbscan_labels).count(-1)

print(f"\n   ✅ DBSCAN terminé:")
print(f"      - Clusters trouvés: {n_clusters_dbscan}")
print(f"      - Points de bruit: {n_noise} ({n_noise/len(dbscan_labels)*100:.2f}%)")

if n_clusters_dbscan > 1:
    # Calculer silhouette score (sans les points de bruit)
    mask_no_noise = dbscan_labels != -1
    if sum(mask_no_noise) > 0 and len(set(dbscan_labels[mask_no_noise])) > 1:
        sil_score = silhouette_score(X_cluster[mask_no_noise], dbscan_labels[mask_no_noise])
        print(f"      - Silhouette Score (sans bruit): {sil_score:.4f}")

# Visualiser DBSCAN sur t-SNE
plt.figure(figsize=(12, 8))
unique_labels = set(dbscan_labels)
colors_dbscan = sns.color_palette('husl', len(unique_labels))

for idx, label in enumerate(unique_labels):
    if label == -1:
        # Bruit en noir
        color = 'black'
        marker_label = 'Bruit'
        alpha = 0.3
        size = 10
    else:
        color = colors_dbscan[idx]
        marker_label = f'Cluster {label}'
        alpha = 0.6
        size = 30
    
    cluster_mask = dbscan_labels == label
    plt.scatter(
        tsne_results[30][cluster_mask, 0],
        tsne_results[30][cluster_mask, 1],
        c=[color],
        label=marker_label,
        alpha=alpha,
        s=size
    )

plt.xlabel('t-SNE 1')
plt.ylabel('t-SNE 2')
plt.title('DBSCAN Clustering')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\n   ✅ DBSCAN clustering terminé")

# 3. HIERARCHICAL CLUSTERING
# ---------------------------
print("\n3️⃣ Hierarchical Clustering")
print("   " + "-" * 50)

# Pour visualiser le dendrogram, utiliser un échantillon plus petit
# (le calcul complet peut être très lent)
sample_size = min(500, len(X_cluster))  # Limiter à 500 échantillons
sample_indices = np.random.choice(len(X_cluster), size=sample_size, replace=False)
X_sample = X_cluster[sample_indices]

print(f"   📊 Calcul de linkage hierarchique sur {sample_size} échantillons...")
# Calculer le linkage
linkage_matrix = linkage(X_sample, method='ward')

# Visualiser le dendrogram
plt.figure(figsize=(15, 6))
dendrogram(
    linkage_matrix,
    truncate_mode='lastp',  # Afficher seulement les dernières p fusions
    p=30,  # Nombre de clusters affichés
    show_leaf_counts=True,
    leaf_font_size=10
)
plt.xlabel('Cluster Index ou (Nombre de points dans le cluster)')
plt.ylabel('Distance')
plt.title('Dendrogram - Hierarchical Clustering (Ward Linkage)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Appliquer Agglomerative Clustering sur toutes les données
print(f"\n   📊 Application d'Agglomerative Clustering avec {k_optimal} clusters...")
hierarchical = AgglomerativeClustering(
    n_clusters=k_optimal,
    linkage='ward'
)
hierarchical_labels = hierarchical.fit_predict(X_cluster)

# Calculer les métriques
sil_score_hier = silhouette_score(X_cluster, hierarchical_labels)
db_score_hier = davies_bouldin_score(X_cluster, hierarchical_labels)
ch_score_hier = calinski_harabasz_score(X_cluster, hierarchical_labels)

print(f"\n   ✅ Hierarchical clustering terminé:")
print(f"      - Silhouette Score: {sil_score_hier:.4f}")
print(f"      - Davies-Bouldin Index: {db_score_hier:.4f}")
print(f"      - Calinski-Harabasz Score: {ch_score_hier:.2f}")

# Visualiser Hierarchical Clustering sur t-SNE
plt.figure(figsize=(12, 8))
colors_hier = sns.color_palette('viridis', k_optimal)

for cluster_id in range(k_optimal):
    cluster_mask = hierarchical_labels == cluster_id
    plt.scatter(
        tsne_results[30][cluster_mask, 0],
        tsne_results[30][cluster_mask, 1],
        c=[colors_hier[cluster_id]],
        label=f'Cluster {cluster_id}',
        alpha=0.6,
        s=30
    )

plt.xlabel('t-SNE 1')
plt.ylabel('t-SNE 2')
plt.title(f'Hierarchical Clustering (k={k_optimal})')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Comparaison des 3 méthodes
print("\n" + "=" * 70)
print("COMPARAISON DES MÉTHODES DE CLUSTERING")
print("=" * 70)
print(f"\n{'Méthode':<20s} {'Silhouette':<15s} {'Davies-Bouldin':<15s} {'Calinski-Harabasz':<20s}")
print("-" * 70)
print(f"{'K-Means':<20s} {silhouette_scores[k_optimal-2]:<15.4f} {davies_bouldin_scores[k_optimal-2]:<15.4f} {calinski_harabasz_scores[k_optimal-2]:<20.2f}")
print(f"{'Hierarchical':<20s} {sil_score_hier:<15.4f} {db_score_hier:<15.4f} {ch_score_hier:<20.2f}")
if n_clusters_dbscan > 1 and 'sil_score' in locals():
    print(f"{'DBSCAN':<20s} {sil_score:<15.4f} {'N/A':<15s} {'N/A':<20s}")
print("\n✅ Analyse de clustering terminée")

## 6. 📈 Tests Statistiques et Extraction de Features

Analyse statistique approfondie et extraction de features texturales avancées.

In [ ]:
# =============================================================================
# TESTS STATISTIQUES ET EXTRACTION DE FEATURES
# =============================================================================

print("=" * 70)
print("TESTS STATISTIQUES ET EXTRACTION DE FEATURES")
print("=" * 70)

# 1. TESTS STATISTIQUES ENTRE CLASSES
# ------------------------------------
print("\n1️⃣ Tests Statistiques entre Classes")
print("   " + "-" * 50)

# Test ANOVA (H0: les moyennes sont égales entre toutes les classes)
# Si p < 0.05, on rejette H0: il y a une différence significative
print("\n   📊 Test ANOVA sur les intensités moyennes par classe...")

# Calculer l'intensité moyenne de chaque image
mean_intensities = np.mean(images, axis=(1, 2, 3))

# Grouper par classe
groups = [mean_intensities[labels_int == i] for i in range(len(config.classes))]

# Test ANOVA
f_statistic, p_value_anova = f_oneway(*groups)
print(f"\n   ANOVA:")
print(f"      - F-statistic: {f_statistic:.4f}")
print(f"      - p-value: {p_value_anova:.6f}")
if p_value_anova < 0.05:
    print(f"      ✅ Différence significative entre les classes (p < 0.05)")
else:
    print(f"      ❌ Pas de différence significative entre les classes (p ≥ 0.05)")

# Test de Kruskal-Wallis (non-paramétrique, alternative à ANOVA)
h_statistic, p_value_kruskal = kruskal(*groups)
print(f"\n   Kruskal-Wallis (test non-paramétrique):")
print(f"      - H-statistic: {h_statistic:.4f}")
print(f"      - p-value: {p_value_kruskal:.6f}")
if p_value_kruskal < 0.05:
    print(f"      ✅ Différence significative entre les classes (p < 0.05)")
else:
    print(f"      ❌ Pas de différence significative entre les classes (p ≥ 0.05)")

# 2. EXTRACTION DE FEATURES TEXTURALES
# -------------------------------------
print("\n2️⃣ Extraction de Features Texturales")
print("   " + "-" * 50)

# Extraire des features sur un échantillon (coûteux en calcul)
n_samples_features = min(100, len(images))  # Limiter pour accélérer
sample_indices_features = np.random.choice(len(images), size=n_samples_features, replace=False)

print(f"\n   📊 Extraction de features sur {n_samples_features} échantillons...")

# Features: contrast, dissimilarity, homogeneity, energy, correlation
features_dict = {
    'contrast': [],
    'dissimilarity': [],
    'homogeneity': [],
    'energy': [],
    'correlation': [],
    'entropy': [],
    'label': []
}

for idx in tqdm(sample_indices_features, desc='   Extraction features'):
    img = images[idx]
    
    # Convertir en uint8 pour GLCM
    img_uint8 = (img * 255).astype('uint8').squeeze()
    
    # Gray-Level Co-occurrence Matrix (GLCM)
    # distances=[1]: pixels adjacents
    # angles=[0, np.pi/4, np.pi/2, 3*np.pi/4]: 4 directions
    glcm = graycomatrix(
        img_uint8,
        distances=[1],
        angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
        levels=256,
        symmetric=True,
        normed=True
    )
    
    # Calculer les propriétés GLCM (moyenne sur les 4 angles)
    features_dict['contrast'].append(np.mean(graycoprops(glcm, 'contrast')))
    features_dict['dissimilarity'].append(np.mean(graycoprops(glcm, 'dissimilarity')))
    features_dict['homogeneity'].append(np.mean(graycoprops(glcm, 'homogeneity')))
    features_dict['energy'].append(np.mean(graycoprops(glcm, 'energy')))
    features_dict['correlation'].append(np.mean(graycoprops(glcm, 'correlation')))
    
    # Entropie de Shannon
    entropy = -np.sum(img * np.log2(img + 1e-10))
    features_dict['entropy'].append(entropy)
    
    features_dict['label'].append(labels_int[idx])

# Créer un DataFrame pour faciliter l'analyse
import pandas as pd
df_features = pd.DataFrame(features_dict)

print(f"\n   ✅ Features extraites: {list(features_dict.keys())[:-1]}")

# Statistiques descriptives par classe
print("\n   📊 Statistiques des features par classe:")
for cls_idx, cls_name in enumerate(config.classes):
    cls_features = df_features[df_features['label'] == cls_idx]
    if len(cls_features) > 0:
        print(f"\n   {cls_name}:")
        print(f"      Contrast:      {cls_features['contrast'].mean():.4f} ± {cls_features['contrast'].std():.4f}")
        print(f"      Homogeneity:   {cls_features['homogeneity'].mean():.4f} ± {cls_features['homogeneity'].std():.4f}")
        print(f"      Energy:        {cls_features['energy'].mean():.4f} ± {cls_features['energy'].std():.4f}")
        print(f"      Entropy:       {cls_features['entropy'].mean():.4f} ± {cls_features['entropy'].std():.4f}")

# Visualiser les distributions des features
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

features_to_plot = ['contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation', 'entropy']
colors = sns.color_palette('husl', len(config.classes))

for idx, feature in enumerate(features_to_plot):
    for cls_idx, cls_name in enumerate(config.classes):
        cls_data = df_features[df_features['label'] == cls_idx][feature]
        if len(cls_data) > 0:
            axes[idx].hist(cls_data, bins=20, alpha=0.5, label=cls_name, color=colors[cls_idx])
    
    axes[idx].set_xlabel(feature.capitalize())
    axes[idx].set_ylabel('Fréquence')
    axes[idx].set_title(f'Distribution: {feature.capitalize()}')
    axes[idx].legend(fontsize=8)
    axes[idx].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Matrice de corrélation des features
print("\n3️⃣ Matrice de Corrélation des Features")
print("   " + "-" * 50)

corr_matrix = df_features[features_to_plot].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.5
)
plt.title('Matrice de Corrélation des Features Texturales')
plt.tight_layout()
plt.show()

print("\n✅ Tests statistiques et extraction de features terminés")

## 7. 📄 Génération du Rapport HTML Automatique

Génération d'un rapport HTML complet avec tous les résultats de l'analyse EDA.

In [ ]:
# =============================================================================
# GÉNÉRATION DU RAPPORT HTML AUTOMATIQUE
# =============================================================================

print("=" * 70)
print("GÉNÉRATION DU RAPPORT HTML")
print("=" * 70)

# Créer le répertoire de sortie pour le rapport
report_dir = config.results_dir / 'eda_report'
report_dir.mkdir(parents=True, exist_ok=True)

print(f"\n📁 Répertoire du rapport: {report_dir}")

# Générer le contenu HTML
html_content = f"""
<!DOCTYPE html>
<html lang='fr'>
<head>
    <meta charset='UTF-8'>
    <meta name='viewport' content='width=device-width, initial-scale=1.0'>
    <title>Rapport EDA - COVID-19 Radiography Dataset</title>
    <style>
        body {{
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            margin: 0;
            padding: 20px;
            background-color: #f5f5f5;
        }}
        .container {{
            max-width: 1200px;
            margin: 0 auto;
            background-color: white;
            padding: 40px;
            box-shadow: 0 2px 10px rgba(0,0,0,0.1);
            border-radius: 8px;
        }}
        h1 {{
            color: #2c3e50;
            border-bottom: 3px solid #3498db;
            padding-bottom: 10px;
        }}
        h2 {{
            color: #34495e;
            margin-top: 40px;
            border-left: 4px solid #3498db;
            padding-left: 15px;
        }}
        h3 {{
            color: #7f8c8d;
        }}
        .info-box {{
            background-color: #ecf0f1;
            padding: 20px;
            border-radius: 5px;
            margin: 20px 0;
        }}
        .metric {{
            display: inline-block;
            margin: 10px 20px 10px 0;
        }}
        .metric-label {{
            font-weight: bold;
            color: #7f8c8d;
        }}
        .metric-value {{
            font-size: 1.2em;
            color: #2c3e50;
        }}
        table {{
            width: 100%;
            border-collapse: collapse;
            margin: 20px 0;
        }}
        th, td {{
            padding: 12px;
            text-align: left;
            border-bottom: 1px solid #ddd;
        }}
        th {{
            background-color: #3498db;
            color: white;
        }}
        tr:hover {{
            background-color: #f5f5f5;
        }}
        .footer {{
            margin-top: 40px;
            padding-top: 20px;
            border-top: 1px solid #ddd;
            text-align: center;
            color: #7f8c8d;
        }}
        .success {{
            color: #27ae60;
        }}
        .warning {{
            color: #e67e22;
        }}
    </style>
</head>
<body>
    <div class='container'>
        <h1>🦠 Rapport d'Analyse Exploratoire Avancée (EDA)</h1>
        <h2>COVID-19 Radiography Dataset</h2>
        
        <div class='info-box'>
            <p><strong>Date de génération:</strong> {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}</p>
            <p><strong>Dataset:</strong> {config.data_dir}</p>
            <p><strong>Classes:</strong> {', '.join(config.classes)}</p>
        </div>
        
        <h2>📊 1. Statistiques du Dataset</h2>
        <div class='info-box'>
            <div class='metric'>
                <span class='metric-label'>Nombre total d'images:</span>
                <span class='metric-value'>{len(images)}</span>
            </div>
            <div class='metric'>
                <span class='metric-label'>Dimensions des images:</span>
                <span class='metric-value'>{images.shape[1]}x{images.shape[2]}</span>
            </div>
            <div class='metric'>
                <span class='metric-label'>Nombre de classes:</span>
                <span class='metric-value'>{len(config.classes)}</span>
            </div>
        </div>
        
        <h3>Distribution des Classes</h3>
        <table>
            <thead>
                <tr>
                    <th>Classe</th>
                    <th>Nombre d'images</th>
                    <th>Pourcentage</th>
                </tr>
            </thead>
            <tbody>
"""

# Ajouter les statistiques par classe
for cls_idx, count in zip(unique_classes, class_counts):
    cls_name = config.classes[cls_idx]
    percentage = (count / len(labels_int)) * 100
    html_content += f"""
                <tr>
                    <td>{cls_name}</td>
                    <td>{count}</td>
                    <td>{percentage:.2f}%</td>
                </tr>
    """

html_content += f"""
            </tbody>
        </table>
        
        <h2>🔬 2. Résultats PCA</h2>
        <div class='info-box'>
            <div class='metric'>
                <span class='metric-label'>Nombre de composantes:</span>
                <span class='metric-value'>{n_components}</span>
            </div>
            <div class='metric'>
                <span class='metric-label'>Variance totale expliquée:</span>
                <span class='metric-value'>{pca.explained_variance_ratio_.sum():.2%}</span>
            </div>
            <div class='metric'>
                <span class='metric-label'>Variance 10 premières PC:</span>
                <span class='metric-value'>{pca.explained_variance_ratio_[:10].sum():.2%}</span>
            </div>
        </div>
        
        <h2>🎯 3. Résultats Clustering</h2>
        
        <h3>K-Means (k={k_optimal})</h3>
        <div class='info-box'>
            <div class='metric'>
                <span class='metric-label'>Silhouette Score:</span>
                <span class='metric-value'>{silhouette_scores[k_optimal-2]:.4f}</span>
            </div>
            <div class='metric'>
                <span class='metric-label'>Davies-Bouldin Index:</span>
                <span class='metric-value'>{davies_bouldin_scores[k_optimal-2]:.4f}</span>
            </div>
            <div class='metric'>
                <span class='metric-label'>Calinski-Harabasz Score:</span>
                <span class='metric-value'>{calinski_harabasz_scores[k_optimal-2]:.2f}</span>
            </div>
        </div>
        
        <h3>DBSCAN</h3>
        <div class='info-box'>
            <div class='metric'>
                <span class='metric-label'>Clusters trouvés:</span>
                <span class='metric-value'>{n_clusters_dbscan}</span>
            </div>
            <div class='metric'>
                <span class='metric-label'>Points de bruit:</span>
                <span class='metric-value'>{n_noise} ({n_noise/len(dbscan_labels)*100:.2f}%)</span>
            </div>
        </div>
        
        <h3>Hierarchical Clustering</h3>
        <div class='info-box'>
            <div class='metric'>
                <span class='metric-label'>Silhouette Score:</span>
                <span class='metric-value'>{sil_score_hier:.4f}</span>
            </div>
            <div class='metric'>
                <span class='metric-label'>Davies-Bouldin Index:</span>
                <span class='metric-value'>{db_score_hier:.4f}</span>
            </div>
            <div class='metric'>
                <span class='metric-label'>Calinski-Harabasz Score:</span>
                <span class='metric-value'>{ch_score_hier:.2f}</span>
            </div>
        </div>
        
        <h2>📈 4. Tests Statistiques</h2>
        
        <h3>Test ANOVA</h3>
        <div class='info-box'>
            <div class='metric'>
                <span class='metric-label'>F-statistic:</span>
                <span class='metric-value'>{f_statistic:.4f}</span>
            </div>
            <div class='metric'>
                <span class='metric-label'>p-value:</span>
                <span class='metric-value {"success" if p_value_anova < 0.05 else "warning"}'>{p_value_anova:.6f}</span>
            </div>
            <div class='metric'>
                <span class='metric-label'>Résultat:</span>
                <span class='metric-value {"success" if p_value_anova < 0.05 else "warning"}'>{'Différence significative (p < 0.05)' if p_value_anova < 0.05 else 'Pas de différence significative'}</span>
            </div>
        </div>
        
        <h3>Test Kruskal-Wallis</h3>
        <div class='info-box'>
            <div class='metric'>
                <span class='metric-label'>H-statistic:</span>
                <span class='metric-value'>{h_statistic:.4f}</span>
            </div>
            <div class='metric'>
                <span class='metric-label'>p-value:</span>
                <span class='metric-value {"success" if p_value_kruskal < 0.05 else "warning"}'>{p_value_kruskal:.6f}</span>
            </div>
            <div class='metric'>
                <span class='metric-label'>Résultat:</span>
                <span class='metric-value {"success" if p_value_kruskal < 0.05 else "warning"}'>{'Différence significative (p < 0.05)' if p_value_kruskal < 0.05 else 'Pas de différence significative'}</span>
            </div>
        </div>
        
        <h2>🎨 5. Features Texturales</h2>
        <p>Features extraites sur {n_samples_features} échantillons aléatoires</p>
        
        <table>
            <thead>
                <tr>
                    <th>Classe</th>
                    <th>Contrast</th>
                    <th>Homogeneity</th>
                    <th>Energy</th>
                    <th>Entropy</th>
                </tr>
            </thead>
            <tbody>
"""

# Ajouter les statistiques des features par classe
for cls_idx, cls_name in enumerate(config.classes):
    cls_features = df_features[df_features['label'] == cls_idx]
    if len(cls_features) > 0:
        html_content += f"""
                <tr>
                    <td>{cls_name}</td>
                    <td>{cls_features['contrast'].mean():.4f} ± {cls_features['contrast'].std():.4f}</td>
                    <td>{cls_features['homogeneity'].mean():.4f} ± {cls_features['homogeneity'].std():.4f}</td>
                    <td>{cls_features['energy'].mean():.4f} ± {cls_features['energy'].std():.4f}</td>
                    <td>{cls_features['entropy'].mean():.4f} ± {cls_features['entropy'].std():.4f}</td>
                </tr>
        """

html_content += f"""
            </tbody>
        </table>
        
        <div class='footer'>
            <p>Rapport généré automatiquement par le notebook Advanced EDA Template</p>
            <p>Data Pipeline - COVID-19 Radiography Analysis</p>
        </div>
    </div>
</body>
</html>
"""

# Sauvegarder le rapport HTML
report_path = report_dir / 'eda_report.html'
with open(report_path, 'w', encoding='utf-8') as f:
    f.write(html_content)

print(f"\n✅ Rapport HTML généré avec succès!")
print(f"   📄 Fichier: {report_path}")
print(f"\n💡 Pour ouvrir le rapport:")
print(f"   - Double-cliquez sur le fichier {report_path.name}")
print(f"   - Ou exécutez: open {report_path} (macOS) / xdg-open {report_path} (Linux)")

# Créer aussi un résumé JSON pour usage programmatique
summary_dict = {
    'generation_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'dataset': {
        'n_images': int(len(images)),
        'image_shape': list(images.shape[1:]),
        'n_classes': len(config.classes),
        'classes': config.classes,
        'class_distribution': {config.classes[idx]: int(count) for idx, count in zip(unique_classes, class_counts)}
    },
    'pca': {
        'n_components': n_components,
        'total_variance_explained': float(pca.explained_variance_ratio_.sum()),
        'variance_top10': float(pca.explained_variance_ratio_[:10].sum())
    },
    'clustering': {
        'kmeans': {
            'n_clusters': k_optimal,
            'silhouette_score': float(silhouette_scores[k_optimal-2]),
            'davies_bouldin_index': float(davies_bouldin_scores[k_optimal-2]),
            'calinski_harabasz_score': float(calinski_harabasz_scores[k_optimal-2])
        },
        'dbscan': {
            'n_clusters': int(n_clusters_dbscan),
            'n_noise_points': int(n_noise)
        },
        'hierarchical': {
            'silhouette_score': float(sil_score_hier),
            'davies_bouldin_index': float(db_score_hier),
            'calinski_harabasz_score': float(ch_score_hier)
        }
    },
    'statistical_tests': {
        'anova': {
            'f_statistic': float(f_statistic),
            'p_value': float(p_value_anova),
            'significant': bool(p_value_anova < 0.05)
        },
        'kruskal_wallis': {
            'h_statistic': float(h_statistic),
            'p_value': float(p_value_kruskal),
            'significant': bool(p_value_kruskal < 0.05)
        }
    }
}

summary_path = report_dir / 'eda_summary.json'
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(summary_dict, f, indent=2, ensure_ascii=False)

print(f"\n✅ Résumé JSON sauvegardé: {summary_path}")

print("\n" + "=" * 70)
print("✅ GÉNÉRATION DU RAPPORT TERMINÉE")
print("=" * 70)

## ✅ Résumé de l'Analyse

### Analyses Réalisées:

1. **📊 Statistiques Descriptives**
   - Distribution des classes
   - Statistiques d'intensité par classe
   - Box plots et histogrammes

2. **🔬 Réduction de Dimensionnalité**
   - PCA avec analyse de variance expliquée
   - t-SNE pour visualisation non-linéaire
   - Projections 2D et 3D

3. **🎯 Clustering**
   - K-Means avec optimisation du nombre de clusters
   - DBSCAN pour clustering basé sur la densité
   - Hierarchical Clustering avec dendrogram
   - Métriques de qualité: Silhouette, Davies-Bouldin, Calinski-Harabasz

4. **📈 Tests Statistiques**
   - Test ANOVA pour comparer les moyennes entre classes
   - Test Kruskal-Wallis (alternative non-paramétrique)
   - Analyse de la significativité statistique

5. **🎨 Features Texturales**
   - Extraction GLCM: Contrast, Homogeneity, Energy, Correlation
   - Entropie de Shannon
   - Analyse de corrélation entre features

6. **📄 Rapport Automatique**
   - Rapport HTML interactif et professionnel
   - Résumé JSON pour usage programmatique
   - Toutes les métriques et statistiques consolidées

---

### 💡 Utilisation du Template:

Ce notebook est un **template réutilisable** que vous pouvez adapter pour d'autres datasets:

1. Modifiez les paramètres dans `config` (nombre d'échantillons, composantes PCA, etc.)
2. Ajustez les chemins vers vos données
3. Personnalisez les visualisations selon vos besoins
4. Le rapport HTML sera automatiquement régénéré avec vos données

### 📚 Prochaines Étapes:

- Entraîner des modèles de classification (CNN, Transfer Learning)
- Appliquer les insights du clustering pour améliorer les modèles
- Utiliser les features extraites pour du Machine Learning classique
- Approfondir l'analyse d'interprétabilité (LIME, SHAP, Grad-CAM)

---

**Auteur:** Data Pipeline Team  
**Date:** Novembre 2025  
**Version:** 1.0
